In [ ]:
import pandas as pd
import numpy as np
# import psycopg2
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, kmeans_plusplus
from scipy.spatial.distance import cdist
from sklearn.metrics import pairwise_distances_argmin
import datetime
from datetime import datetime
import portutils as utils
import ffn

from custom_data_loader import CustomDataLoader, ExpandingScaler, feature_engineer, filter_date_range


from jumpmodels. jump import JumpModel
from jumpmodels.utils import filter_date_range
# from feature_modified import DataLoader
from jumpmodels.preprocess import StandardScalerPD, DataClipperStd
from jumpmodels.plot import plot_regimes_and_cumret, savefig_plt, plot_regimes

In [ ]:
class StatJumpModel:
    def __init__ (
            self,
            n_states=2, 
            lambda_penalty=0, 
            max_iter=50, 
            scale_features=True, 
            clip=True,
            n_init=10,
            random_state=None,
            tol=1e-8,
            min_periods=10,
    ):
        self.n_states = n_states
        self.lambda_penalty = lambda_penalty
        self.max_iter = max_iter
        self.scale_features = scale_features
        self.clip = clip
        self.n_init = n_init
        self.random_state = random_state
        self.tol = tol
        self.min_periods = min_periods

        self.state_sequence = None
        self.theta = None

    def _loss(self, y, theta):
        return 0.5 * np.linalg.norm(y - theta) ** 2
    
    def expanding_standard_scaler(self, df, feature_cols):
        scaled_features = pd.DataFrame(index=df.index)
        for col in feature_cols:
            # expanding_mean = df[col].expanding(min_periods=min_periods).mean()
            # expanding_std = df[col].expanding(min_periods=min_periods).std()
            scaled_features[col] = df[col].expanding(min_periods=self.min_periods).apply(lambda x: (x[-1] -x.mean()) / x.std())
            if (self.clip==True):
                scaled_features[col]=scaled_features[col].clip(-3,3)

        return scaled_features.dropna(inplace=False)
    
    def _kmeans_initialization(self, features, random_state=None):
        theta_init, indices = kmeans_plusplus(X=features, n_clusters=self.n_states, random_state=random_state)
        theta_init= theta_init[theta_init[:, 3].argsort()[::-1]]## would it or should it always be 3? can we check how state labeling in JM?
        state_seq_init = pairwise_distances_argmin (features, theta_init)

        return theta_init, state_seq_init
    
    def _update_parameters(self, features, state_seq):
        theta_new = np.zeros ((self.n_states, features.shape[1]))
        for state in range(self.n_states):
            assigned_features = features[state_seq == state]
            theta_new[state] = assigned_features.mean(axis=0)
        return theta_new
    
    def _update_states(self, features, theta, online=True):
        T = features.shape[0]
        K = self.n_states
        lambda_penalty = self.lambda_penalty

        cost = np.zeros((T, K))
        cost = 0.5 * cdist(features, theta, 'sqeuclidean')

        V = np.full((T, K), fill_value=np.inf)
        path = np.zeros((T, K), dtype=int)

        V[0] = cost[0]

        for t in range(1, T):
            for k in range(K):
                transition_cost = V[t-1] + (lambda_penalty * (1 - np.eye(K)[k]))
                min_prev_state = np.argmin(transition_cost)
                V[t, k] = cost[t, k] + transition_cost[min_prev_state]
                path[t, k] = min_prev_state

        if online==True:
            return np.argmin(V, axis=1)

        state_seq_new = np.zeros(T, dtype=int)
        state_seq_new[T-1] = np.argmin(V[T-1])

        for t in range(T-2, -1, -1):
            state_seq_new[t] = path[t+1, state_seq_new[t+1]]

        # V[T-1] = cost[T-1]
        # for t in range(T-1, -1, -1):
        #     for k in range(K):
        #         transition_cost = V[t+1] + (lambda_penalty * (1 - np.eye(K)[k]))
        #         min_next_state = np.argmin(transition_cost)
        #         V[t, k] = cost[t, k] + transition_cost[min_next_state]
        #         path[t, k] = min_next_state

        # state_seq_new = np.zeros(T, dtype=int)
        # state_seq_new[0] = np.argmin(V[0])
        # for t in range(1, T):
        #     state_seq_new[t] = path[t-1, state_seq_new[t-1]]

        return state_seq_new
    
    def _objective_function(self, features, theta, state_seq):
        fit_cost = np.sum([self._loss(features[t], theta[state_seq[t]]) for t in range(len(features))])
        jump_cost = np.sum([1 for t in range(1, len(state_seq)) if state_seq[t] != state_seq[t-1]])
        return fit_cost + jump_cost
    
    def fit(self, df, feature_cols):

        if self.scale_features:
            scaled_features = self. _expanding_standard_scaler(df, feature_cols)
            df_scaled = df.copy()
            df_scaled[feature_cols] = scaled_features[feature_cols]
            df_scaled.dropna(inplace=False)
            features = df_scaled[feature_cols].values
        else:
            features = df[feature_cols].values

        best_obj = np.inf
        best_theta = None
        best_state_seq = None

        for init in range(self.n_init):
            random_state = self.random_state + init if self.random_state is not None else None
            theta_init, state_seq_init = self._kmeans_initialization(features, random_state=random_state)

            theta = theta_init.copy()
            state_seq = self._update_states(features, theta, online=False)
            obj = self._objective_function(features, theta, state_seq)

            for interation in range(self.max_iter):
                theta_new = self._update_parameters(features, state_seq)
                state_seq_new = self._update_states(features, theta_new, online=False)
                new_obj = self._objective_function(features, theta_new, state_seq_new)

                if (obj - new_obj) > self.tol:
                    theta = theta_new.copy()
                    state_seq = state_seq_new.copy()
                    obj = new_obj
                else:
                    break

            if obj < best_obj:
                best_obj = obj
                best_theta = theta.copy()
                best_state_seq = state_seq.copy()

        self.theta = best_theta 
        self.state_sequence = best_state_seq

        return self
    
    def predict(self, df, feature_columns, online=True):
        if self.theta is None:
            raise ValueError("Model not fitted yet. Please fit the model first.")
        
        if self.scale_features:
            scaled_features = self. _expanding_standard_scaler(df, feature_columns)
            df_scaled = df.copy()
            df_scaled[feature_columns] = scaled_features[feature_columns]
            df_scaled.dropna(inplace=True)
            features = df_scaled[feature_columns].values
        else:
            features = df[feature_columns].values

        state_seq = self._update_states(features, self.theta, online=online)
        return state_seq
    
    def fit_predict(self, df, feature_columns):
        self.fit(df, feature_columns)
        return self.predict(df, feature_columns)
    
    def assign_labels(self):
        bull_state = np.argmax(self.theta[:, 3])
        bear_state = 1 - bull_state
        return {bull_state: "Bull", bear_state: "Bear"}

In [ ]:
# CUSTOM DATA LOADER
ret_ser = CustomDataLoader(ticker="Nifty 50 TR").load(start_date=None, end_date=None)

train_start, test_start = ret_ser.index[0], '2020-01-01'
train_end, test_end = '2019-12-31', ret_ser.index[-1]

ret_ser_train = filter_date_range(ret_ser, train_start, train_end)
ret_ser_test = filter_date_range(ret_ser, test_start, test_end)


X_train= feature_engineer(ret_ser_train)
X_test= feature_engineer (ret_ser_test)

custom_clipper = DataClipperStd(mul=3.)
custom_scaler = StandardScalerPD()

custom_X_train_processed = custom_scaler.fit_transform(custom_clipper.fit_transform(X_train))
custom_X_test_processed = custom_scaler.transfpgrm(custom_clipper.transform(X_test))

In [ ]:
#### RUNNING BOTH JUMP MODELS ON NIFTY 50 DATA FROM CUSTOM DATA LOADER ###

ver='v0'
lambda_penalty = 50
max_iter = 100
n_init = 10
random_state = 42
min_periods = 30

if ver=='v0':
    feature_cols_custom = ['ret_20', 'DD-log_20', 'sortino_20', 'ret_60', 'DD-log_60', 'sortino_60', 'ret_120', 'DD-log_120', 'sortino_120']
else:
    feature_cols_custom = ['DD_20', 'DD_20_60_diff', 'DD_60_120_diff', 'ret_120']

# CUSTOM STATJUMP MODEL
model_custom = StatJumpModel(
    n_states=2,
    lambda_penalty = lambda_penalty,
    max_iter=max_iter, 
    scale_features=False, 
    clip=False, 
    n_init=n_init,
    random_state=random_state, 
    min_periods=min_periods
)

model_custom.fit(custom_X_train_processed, feature_cols_custom)
nifty50_state_sequence = model_custom.predict(custom_X_test_processed, feature_cols_custom)

state_label_mapping = model_custom.assign_labels()
nifty50_state_sequence_indexed = pd.Series(nifty50_state_sequence, index=custom_X_test_processed.index)